#  ⭐ `dim_forma_farmaceutica`


In [1]:
%run ../../config/bootstrap.py

import pandas as pd
import numpy as np
import re 
from unidecode import unidecode
from rapidfuzz import process, fuzz


from pathlib import Path
from utils import get_project_root, criar_dim_forma_farmaceutica, normalizar_forma_farmaceutica_raw

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(path) 
bronze.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO'],
      dtype='object')

In [4]:
bronze.head(2)

,IDENTIFICACAO_NOTIFICACAO,RELACAO_MEDICAMENTO_EVENTO,NOME_MEDICAMENTO_WHODRUG,PRINCIPIOS_ATIVOS_WHODRUG,CODIGO_ATC,DETENTOR_REGISTRO,CONCENTRACAO,COMPONENTE_SUSPEITO,ACAO_ADOTADA,PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO,INDICACAO_MEDDRA,INDICACAO_RELATADA_NOTIFICADOR_INICIAL,DOSE,FREQUENCIA_DOSE,POSOLOGIA,DURACAO,INICIO_ADMINISTRACAO,FIM_ADMINISTRACAO,FORMA_FARMACEUTICA,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_MAE_PAI,NUMELO_LOTE,ATUALIZADO
0,BR-ANVISA-300000004,Suspeito,Oxacilina,Oxacillin sodium,J01CF,Teuto,None,None,None,None,None,None,250 milligram (mg),6 hora,None,4 dia,20181124,None,None,infusão intravenosa,None,5833018,2025-11-17
1,BR-ANVISA-300000005,Suspeito,Paracemol,Paracetamol,N02BE,None,None,None,Retirada do medicamento,None,None,None,None,None,None,None,20181122,20181122,None,oral,None,None,2025-11-17


In [5]:
bronze.FORMA_FARMACEUTICA.nunique()

4494

In [6]:
bronze.FORMA_FARMACEUTICA.value_counts().head(20)

FORMA_FARMACEUTICA
Tablet                                   9792
comprimido                               7291
Solução injetável                        7177
solução injetável                        7077
Solution for injection                   6717
Comprimido                               4215
ampola                                   3911
Film-coated tablet                       3827
AMPOLA                                   3617
Injetável                                3084
Concentrate for solution for infusion    2903
Solution for infusion                    2863
SOLUTION FOR INJECTION                   2680
Unknown                                  2611
COMPRIMIDO                               2425
Ampola                                   2334
injetável                                2204
SOLUÇÃO INJETÁVEL                        1981
solução                                  1822
injetavel                                1713
Name: count, dtype: int64

# 🥈 Silver

normalized data, medium quality


In [7]:
silver = bronze[['FORMA_FARMACEUTICA']].drop_duplicates().fillna('Desconhecida')

In [8]:
silver.head()

,FORMA_FARMACEUTICA
0,Desconhecida
7,solução injetável
861,Solução injetável
996,COMPRIMIDO SIMPLES
1000,Comprimido simples


In [9]:
silver.FORMA_FARMACEUTICA.nunique()

4494

## hist_dim_forma_farmaceutica

In [10]:
hist_dim_forma_farmaceutica = silver[['FORMA_FARMACEUTICA']].drop_duplicates().fillna('Desconhecido')


In [11]:
hist_dim_forma_farmaceutica, dim_forma_farmaceutica = criar_dim_forma_farmaceutica(hist_dim_forma_farmaceutica, col_original="FORMA_FARMACEUTICA")


In [12]:
hist_dim_forma_farmaceutica.head()

,FORMA_FARMACEUTICA,FORMA_FARMACEUTICA_CHAVE,FORMA_FARMACEUTICA_VALOR
0,Desconhecida,Desconhecida,1
1,solução injetável,Solucao injetavel,2
2,Solução injetável,Solucao injetavel,2
3,COMPRIMIDO SIMPLES,Comprimido,3
4,Comprimido simples,Comprimido,3


In [13]:
dim_forma_farmaceutica

,FORMA_FARMACEUTICA,FORMA_FARMACEUTICA_VALOR
0,Desconhecida,1
1,Solucao injetavel,2
2,Comprimido,3
3,Outros,4
4,Capsula,5
5,Solucao oral,6
6,Suspensao oral,7
7,Pomada,8
8,Po para solucao injetavel,9
9,Gotas orais,10


In [14]:
hist_dim_forma_farmaceutica.to_parquet(Path(project_root) / "data/02_silver/hist_dim_forma_farmaceutica/hist_dim_forma_farmaceutica.parquet", index=False)

In [17]:
hist_dim_forma_farmaceutica.to_csv(Path(project_root) / "data/02_silver/hist_dim_forma_farmaceutica/MANUAL_hist_dim_forma_farmaceutica.csv", sep=";", index=False)

# 🥇 Gold



## dim_forma_farmaceutica

In [ ]:
dim = hist_dim_forma_farmaceutica[['FORMA_FARMACEUTICA_CHAVE','FORMA_FARMACEUTICA_VALOR']].copy()
dim.to_parquet(Path(project_root) / "data/03_gold/dim_forma_farmaceutica/dim_forma_farmaceutica.parquet", index=False)

In [ ]:
dim = hist_dim_forma_farmaceutica[['FORMA_FARMACEUTICA_CHAVE','FORMA_FARMACEUTICA_VALOR']].copy()
dim.to_parquet(Path(project_root) / "data/03_gold/dim_forma_farmaceutica/dim_forma_farmaceutica.parquet", index=False)